# Silver Layer — CRM: Product Info

Transforms the bronze `workspace.bronze.prd_info` table into `workspace.silver.product_info` with meaningful, readable column names.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Column Renaming** | Renamed abbreviated columns to meaningful names (e.g., `prd_id` → `product_id`, `prd_nm` → `product_name`) |
| 2 | **Trim Whitespace** | Trimmed leading/trailing spaces from `product_name` and `product_line` |
| 3 | **Product Line Mapping** | Mapped codes to full words: `R` → `Road`, `S` → `Sport`, `M` → `Mountain`, `T` → `Touring`, `NULL` → `Unknown` |
| 4 | **NULL Cost Handling** | Replaced NULL `cost` values with `0` for safe downstream aggregations |
| 5 | **NULL End Date Handling** | Added `is_active` boolean column (`True` when `end_date` is NULL, `False` otherwise) |
| 6 | **Category Key Extraction** | Added `Category_key` column — first 5 characters of `product_key` with hyphens replaced by underscores (e.g., `CO-RF-FR-R92B-58` → `CO_RF`) |

---

## Column Mapping (Bronze → Silver)

| Bronze Column | Silver Column | Transformation Applied |
|---|---|---|
| `prd_id` | `product_id` | Renamed |
| `prd_key` | `product_key` | Renamed |
| `prd_key` | `Category_key` | New column: first 5 characters of `prd_key` via `substr(1, 5)`, hyphens replaced with underscores |
| `prd_nm` | `product_name` | Renamed + Trimmed |
| `prd_cost` | `cost` | Renamed + NULLs replaced with `0` |
| `prd_line` | `product_line` | Renamed + Trimmed + Mapped (`R`→`Road`, `S`→`Sport`, `M`→`Mountain`, `T`→`Touring`, `NULL`→`Unknown`) |
| `prd_start_dt` | `start_date` | Renamed |
| `prd_end_dt` | `end_date` | Renamed (NULL kept = still active) |
| — | `is_active` | New column: `True` if `end_date` is NULL, `False` otherwise |

---

## Implementation

All transformations are implemented in **PySpark** (`pyspark.sql.functions`):
- `col()` — select and alias columns
- `trim()` — remove leading/trailing whitespace
- `when()` + `otherwise()` — conditional mapping for product_line and is_active
- `coalesce()` — replace NULL cost with 0
- `substr()` — extract first 5 characters of `prd_key` for `Category_key`
- `regexp_replace()` — replace hyphens with underscores in `Category_key`
- `df.write.mode("overwrite").saveAsTable()` — write to silver layer

In [0]:
from pyspark.sql.functions import col, trim, when, coalesce, lit, regexp_replace

df = (spark.read.table("workspace.bronze.prd_info")
    .select(
        col("prd_id").alias("product_id"),
        col("prd_key").alias("product_key"),
        regexp_replace(col("prd_key").substr(1, 5), "-", "_").alias("Category_key"),
        trim(col("prd_nm")).alias("product_name"),
        coalesce(col("prd_cost"), lit(0)).alias("cost"),
        when(trim(col("prd_line")) == "R", "Road")
            .when(trim(col("prd_line")) == "S", "Sport")
            .when(trim(col("prd_line")) == "M", "Mountain")
            .when(trim(col("prd_line")) == "T", "Touring")
            .otherwise("Unknown").alias("product_line"),
        col("prd_start_dt").alias("start_date"),
        col("prd_end_dt").alias("end_date"),
        when(col("prd_end_dt").isNull(), True).otherwise(False).alias("is_active"),
    )
)

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.product_info")

In [0]:
display(spark.read.table("workspace.silver.product_info").limit(10))